# Contagion engine: stress test demo

1. Set a macro shock in natural units.
2. Read the ranked table. The **hybrid engine** gives the ranking: a seasonal ridge base path plus the accounting transmission of the shock to the debt of each company. The **neural engine** (the graph recurrent model) is shown next to it, for comparison.
3. Select a ticker and read its audit trail.

Before you run this notebook, build the data and train the models (see `README.md`).

In [1]:
import os, sys, json, warnings
warnings.filterwarnings('ignore')
root = os.path.abspath('..') if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
os.chdir(root); sys.path.insert(0, root)
import pandas as pd
from src.simulator.stress_test import simulate
from src.knowledge.audit import Auditor
pd.set_option('display.float_format', lambda value: f'{value:,.2f}')
pd.set_option('display.width', 200)

## 1. How good is each engine?

1-step forecast error on the test split (2022-Q1 to 2024-Q4), divided by the usual quarterly change of each company. Lower is better. `mean_change` is the naive forecast that both engines start from.

In [2]:
metrics = json.load(open('results/metrics.json'))
rows = {f'baseline {k}': v for k, v in metrics['baselines'].items()}
rows['hybrid ridge'] = metrics['ridge']
rows.update({f'neural {k}': v for k, v in metrics['models'].items()})
pd.DataFrame(rows).T[['mse_std', 'mae_std', 'median_abs_std', 'train_mse_std']]

,mse_std,mae_std,median_abs_std,train_mse_std
baseline no_change,7.36,1.27,0.67,1.08
baseline mean_change,7.05,1.21,0.65,0.97
hybrid ridge,6.96,1.20,0.63,0.96
neural sector,6.98,1.21,0.65,0.95
neural no_edges,7.09,1.22,0.65,0.95
neural typed,6.97,1.21,0.65,0.95


## 2. The shock

- `SELIC_delta`: percentage points of the SELIC target. `3.0` is +300 basis points.
- `IPCA_delta`: percentage points of the quarterly IPCA change.
- `PTAX_delta_pct`: percent change of the USD/BRL rate. `15` is +15 %.

The shock starts after the last filed quarter and holds for 4 quarters. `hybrid_change_pct` and `neural_change_pct` are the equity change caused by the shock, as a percent of the last filed equity.

In [3]:
shocks = {'SELIC_delta': 3.0, 'IPCA_delta': 0.0, 'PTAX_delta_pct': 15.0}
scenario = simulate(shocks, horizon=4)
table = scenario.table()
columns = ['rank', 'ticker', 'sector', 'equity_filed', 'hybrid_change_pct', 'neural_change_pct', 'neural_rank',
           'floating_share', 'floating_basis', 'coverage_under_shock', 'breaching_suppliers']
table[columns].head(15)

,rank,ticker,sector,equity_filed,hybrid_change_pct,neural_change_pct,neural_rank,floating_share,floating_basis,coverage_under_shock,breaching_suppliers
0,1,BRKM5,Petroquímicos e Borracha,"-4,277,999,976.40",NaN,NaN,<NA>,0.04,extracted,-0.05,PETR4
1,2,GOLL4,Serviços Transporte e Logística,"-29,090,518,435.28",NaN,NaN,<NA>,NaN,none,NaN,
2,3,OIBR3,Telecomunicações,"-16,286,601,673.79",NaN,NaN,<NA>,0.08,extracted,-0.21,
3,4,RCSL3,"Máquinas, Equipamentos, Veículos e Peças","-17,476,999.32",NaN,NaN,<NA>,0.00,none,-3.06,
4,5,BEEF3,Alimentos,"87,216,997.73","-2,191.82",-17.34,5,0.61,extracted,0.43,
5,6,KLBN11,Papel e Celulose,"8,637,199,867.86",-39.78,-0.48,16,0.08,extracted,1.10,
6,7,SUZB3,Papel e Celulose,"32,415,574,419.12",-25.56,-0.24,34,0.03,extracted,0.51,
7,8,JBSS3,Alimentos,"50,370,425,010.15",-21.13,-0.33,21,0.00,extracted,1.85,
8,9,LEVE3,"Máquinas, Equipamentos, Veículos e Peças","770,596,976.24",-12.21,-0.54,11,0.00,extracted,1.38,
9,10,MYPK3,"Máquinas, Equipamentos, Veículos e Peças","5,070,982,114.92",-8.11,-0.25,31,0.41,extracted,1.09,


## 3. Checks

Each predicted balance sheet must balance. A zero shock must give a zero change in both engines.

In [4]:
print('rank correlation, hybrid vs neural:', round(scenario.rank_correlation(), 2))
print('largest balance residual / assets:', f"hybrid {table['hybrid_residual'].max():.2e}, neural {table['neural_residual'].max():.2e}")
zero = simulate({}, horizon=4).table()
print('largest change with a zero shock (%):', float(zero['hybrid_change_pct'].abs().max()), float(zero['neural_change_pct'].abs().max()))

rank correlation, hybrid vs neural: 0.47
largest balance residual / assets: hybrid 6.52e-08, neural 2.03e-07


largest change with a zero shock (%): 0.0 0.0


## 4. The audit trail of 1 company

Each line says where its numbers come from: `input`, `filed`, `extracted`, `rule`, `model`, `estimate` or `none`.

In [5]:
auditor = Auditor(scenario)
ticker = table.loc[~table['equity_not_positive']].iloc[0]['ticker']  # change this to any ticker of the table
print(auditor.render(ticker))

BEEF3 | MINERVA S/A | Alimentos
hybrid rank 5 of 93 ranked companies | equity change caused by the shock: -2191.82 % of filed equity
 1. [input    ] shock: SELIC +3.00 pp, IPCA +0.00 pp, PTAX +15.0 %, held for 4 quarters after 2024-12-31
 2. [filed    ] last filed balance: total_assets R$ 42.14 bn, current_liabilities R$ 16.57 bn, noncurrent_liabilities R$ 25.48 bn, equity R$ 87.2 mn
 3. [extracted] debt exposure: gross debt R$ 30.08 bn (filed accounts 2.01.04 + 2.02.01); floating R$ 23.26 bn (61%) [extracted], IPCA R$ 0.0 mn (0%), foreign currency not swapped R$ 14.66 bn (39%) [extracted]; source: debt note, unit x1,000 (stated)
 4. [filed    ] FX cross-check: foreign-currency lines of the note R$ 25.19 bn (swapped or not) against the filed split 'Em Moeda Estrangeira' (2.01.04.01.02 + 2.02.01.01.02) R$ 23.01 bn: ok. A flag is shown only; it does not change the shock.
 5. [rule     ] transmission, quarter +1: extra interest R$ 174.5 mn, IPCA accrual R$ 0.0 mn, FX revaluation R$ 2.20 b

In [6]:
print(auditor.render('PETR4'))

PETR4 | PETRÓLEO BRASILEIRO S.A. - PETROBRAS | Petróleo e Gás
hybrid rank 21 of 93 ranked companies | equity change caused by the shock: -2.99 % of filed equity
 1. [input    ] shock: SELIC +3.00 pp, IPCA +0.00 pp, PTAX +15.0 %, held for 4 quarters after 2024-12-31
 2. [filed    ] last filed balance: total_assets R$ 1,124.80 bn, current_liabilities R$ 194.81 bn, noncurrent_liabilities R$ 562.48 bn, equity R$ 367.51 bn
 3. [extracted] debt exposure: gross debt R$ 373.47 bn (filed accounts 2.01.04 + 2.02.01); floating R$ 0.0 mn (0%) [extracted], IPCA R$ 0.0 mn (0%), foreign currency not swapped R$ 110.82 bn (77%) [extracted]; source: debt note, unit x1,000,000 (stated)
 4. [filed    ] FX cross-check: the filing leaves the split 'Em Moeda Estrangeira' at 0, so the check does not run
 5. [rule     ] transmission, quarter +1: extra interest R$ 0.0 mn, IPCA accrual R$ 0.0 mn, FX revaluation R$ 16.62 bn; after 34% tax, equity R$ -10.97 bn
 6. [rule     ] transmission, quarter +2: extra intere

## Limits

- The hybrid shock effect is an accounting pass-through with stated assumptions (34 % tax, no change in revenue or costs). It is not learned from data.
- The neural shock response is an extrapolation from 45 quarters of macro data. The neural engine does not beat the naive forecast on the test split.
- The universe holds companies with all quarters from 2010-Q4 to 2024-Q4 (survivors). Statements are the latest filed version, not point-in-time.
- The extraction covers only what a company discloses. An attention weight is not proof of a causal link.